In [3]:
import pandas as pd
df = pd.read_csv("combined_dataset.csv", encoding='utf-8')
print(df.head())
print(df['hate'].value_counts())

                                                text  hate
0   Are bhajans played daily.??  5 times a day..?? 🤔     1
1  @zainabsikander Perfectly said.... 100% agreed...     1
2  Tats sad no mother thinks her baby as puppy bu...     1
3  Speechs wont work if ill thoughts still around...     1
4  @timesofindia Mamta is doing wrong. But where ...     0
hate
0    9507
1    9497
Name: count, dtype: int64


In [ ]:
import re
import pandas as pd
import contractions

def clean_text(text):
    text = contractions.fix(text)
    text = re.sub(r'@\w+', '[Identity]', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = text.lower()
    return text

df = pd.read_csv("combined_dataset.csv")

df['cleaned_text'] = df['text'].apply(clean_text)

with open("hinglish_corpus.txt", "w", encoding="utf8") as f:
    for text in df['cleaned_text']:
        if text.strip():
            f.write(text.strip() + "\n")

df = pd.read_csv("hinglish_corpus.txt", encoding='utf-8')
print(df.head())
print(df['hate'].value_counts())

                                                text  hate  \
0   Are bhajans played daily.??  5 times a day..?? 🤔     1   
1  @zainabsikander Perfectly said.... 100% agreed...     1   
2  Tats sad no mother thinks her baby as puppy bu...     1   
3  Speechs wont work if ill thoughts still around...     1   
4  @timesofindia Mamta is doing wrong. But where ...     0   

                                        cleaned_text  
0              are bhajans played daily  times a day  
1  identity perfectly said  agreed to gavaskar si...  
2  tats sad no mother thinks her baby as puppy bu...  
3  speechs will not work if ill thoughts still ar...  
4  identity mamta is doing wrong but where was i ...  
hate
0    9507
1    9497
Name: count, dtype: int64


In [5]:

import pandas as pd
import contractions
import string
import re
import time
import nltk
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, RandomizedSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [9]:
nltk.download('wordnet')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = contractions.fix(text)
    text = re.sub(r'@\w+', '[IDENTITY]', text)
    # text = re.sub(r'#\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = text.lower()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    return " ".join(tokens)


df['cleaned_text'] = df['text'].apply(clean_text)
print(df.head)
# Feature extraction: BoW(binary)
vectorizer = CountVectorizer(ngram_range=(1,4),binary=True)
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['hate']

# Feature selection using Mutual information classification
selector = SelectKBest(score_func=chi2, k=50000)
X_selected = selector.fit_transform(X, y)

from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in sss.split(X_selected, y):
    X_train, X_test = X_selected[train_idx], X_selected[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Model training with GridSearch for smoothing (epsilon / alpha)
parameters = {'alpha': [0.05]}
nb = MultinomialNB()
clf = GridSearchCV(nb, parameters, cv=5)
clf.fit(X_train, y_train)
best_model = clf.best_estimator_

# Evaluation
start = time.time()
y_pred = best_model.predict(X_test)
end = time.time()

print("Best alpha (ε):", clf.best_params_['alpha'])
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=['Not Hate', 'Hate']))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(f"Testing time per statement: {(end - start) / len(y_test) * 1e6:.4f} µs")

# 10-fold cross-validation
cv_scores = cross_val_score(best_model, X_selected, y, cv=10)
print("10-fold CV Accuracy: %.4f" % cv_scores.mean())
print("95%% CI: (%.4f, %.4f)" % (cv_scores.mean() - 2 * cv_scores.std(), cv_scores.mean() + 2 * cv_scores.std()))


[nltk_data] Downloading package wordnet to /Users/akki/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/akki/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


<bound method NDFrame.head of                                                     text  hate  \
0       Are bhajans played daily.??  5 times a day..?? 🤔     1   
1      @zainabsikander Perfectly said.... 100% agreed...     1   
2      Tats sad no mother thinks her baby as puppy bu...     1   
3      Speechs wont work if ill thoughts still around...     1   
4      @timesofindia Mamta is doing wrong. But where ...     0   
...                                                  ...   ...   
18999  @CityNews @Twitter I think he's done quite eno...     1   
19000  Binoy! u know nothing about him. update ur kno...     0   
19001  Employment opportunities for youth is there in...     0   
19002  Stupid statement from an actor. Priyanca is sm...     1   
19003  Only 1300 cases of tripple talaq have been fil...     0   

                                            cleaned_text  
0                    are bhajans played daily time a day  
1      identity perfectly said agreed to gavaskar sir...  
